# Decision Assurance Layer — Incident Walkthrough

**IBM AI Builders Challenge — July Wildcard**

---

This notebook walks through a complete simulated cybersecurity incident using the
Decision Assurance Layer (DAL) directly — without running the Streamlit UI.

It demonstrates:
1. Loading a threat scenario from the fixture library
2. Generating an AI recommendation via IBM watsonx.ai (Granite) — or mock in demo mode
3. Generating a ranked action analysis across all three response options
4. Submitting an analyst decision with a documented rationale
5. Verifying the saved record's SHAKE-256 integrity hash
6. Simulating a tampered record and observing the mismatch detection
7. Reviewing the full audit log

**No IBM credentials required** — the DAL runs in demo mode automatically when
`.env` is absent, using realistic mock AI responses and local file storage.

---

## Setup

Run from the repo root:
```bash
pip install -r requirements.txt
pip install jupyter
jupyter notebook notebooks/dal_walkthrough.ipynb
```

In [ ]:
import sys
import os
import json
from pathlib import Path

# Ensure the repo root is on the path when running from notebooks/
repo_root = Path(".").resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

from src import scenario_loader, dal_engine, cos_client, watsonx_client
from src.decision_record import DecisionRecord

print("DAL modules loaded.")
print(f"AI mode    : {'Demo/Mock' if watsonx_client.is_demo_mode() else 'IBM watsonx.ai (Granite)'}")
print(f"Storage    : {'Local (data/decisions/)' if cos_client.is_demo_mode() else 'IBM Cloud Object Storage'}")
print(f"Scenarios  : {len(scenario_loader.list_scenarios())} loaded")

---
## Step 1 — Choose a Scenario

The DAL ships with 10 realistic threat scenario fixtures covering a range of attack
types and severities. We will work through the **Supply Chain Attack** scenario — a
CRITICAL-severity incident involving a malicious package in the CI/CD pipeline.

In [ ]:
# List all available scenarios
print("Available scenarios:\n")
for s in scenario_loader.list_scenarios():
    print(f"  [{s['severity']:8}]  {s['id']}  —  {s['title']}")

In [ ]:
SCENARIO_ID = "scenario_007"   # Supply Chain Attack — change to any scenario_00N

scenario = scenario_loader.get_scenario(SCENARIO_ID)
print(f"Title   : {scenario['title']}")
print(f"Severity: {scenario['severity']}")
print(f"MITRE   : {scenario['mitre_tactic']}  /  {scenario['mitre_technique']}")
print(f"Host    : {scenario['affected_host']}")
print(f"Time    : {scenario['timestamp']}")
print("\nRaw Evidence:")
for i, entry in enumerate(scenario['raw_evidence'], 1):
    print(f"  {i}. {entry}")

---
## Step 2 — Generate an AI Recommendation

The DAL sends the scenario to IBM watsonx.ai (Granite) and receives a structured
recommendation: a top action, a confidence score, reasoning, and suggested steps.

This is **Element 2 (AI Recommendation)** and **Element 3 (Uncertainty Score)**
of the Decision Assurance Framework.

In [ ]:
result = dal_engine.get_recommendation(SCENARIO_ID)

print(f"Recommendation : {result['recommendation'].upper()}")
print(f"Confidence     : {result['confidence_score']:.0%}  (source: {result['source']})")
print(f"\nReasoning:\n  {result['reasoning']}")
print("\nSuggested Actions:")
for i, action in enumerate(result['suggested_actions'], 1):
    print(f"  {i}. {action}")

---
## Step 3 — Ranked Action Analysis

The ranked analysis scores **all three** possible response actions independently:
escalate, investigate, and dismiss. This is the full decision landscape —
the AI's view of how appropriate each option is for this specific alert.

Low scores for investigate and dismiss at high severity demonstrate why the
analyst should feel confident escalating — and why the AI's uncertainty on the
top action still leaves room for human override.

In [ ]:
ranked = dal_engine.get_ranked_actions(SCENARIO_ID)

print(f"Ranked actions for: {scenario['title']}\n")
for i, item in enumerate(ranked['ranked_actions'], 1):
    bar = '█' * int(item['confidence_score'] * 20)
    print(f"  #{i}  {item['action'].upper():11}  {item['confidence_score']:.0%}  {bar}")
    print(f"       {item['reasoning']}")
    for step in item['suggested_steps']:
        print(f"         • {step}")
    print()

In [ ]:
# Visualise with a simple bar chart if matplotlib is available
try:
    import matplotlib.pyplot as plt
    actions = [a['action'].capitalize() for a in ranked['ranked_actions']]
    scores  = [a['confidence_score'] for a in ranked['ranked_actions']]
    colours = ['#ef4444' if a == 'escalate' else '#f59e0b' if a == 'investigate' else '#22c55e'
               for a in [a['action'] for a in ranked['ranked_actions']]]
    fig, ax = plt.subplots(figsize=(6, 3))
    bars = ax.barh(actions, scores, color=colours, height=0.5)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Confidence Score')
    ax.set_title(f'Ranked Action Confidence — {scenario["alert_type"]}')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.0%}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping chart. Run: pip install matplotlib")

---
## Step 4 — Analyst Decision

This is **Element 4 (Human Judgment)** of the Decision Assurance Framework.

The analyst reviews the evidence and AI output, then makes an explicit decision:
**Approve**, **Reject**, or **Override** — with a mandatory documented rationale.

The rationale is not optional. It is the record that demonstrates the human was
genuinely in the loop — not just rubber-stamping the AI.

In [ ]:
# The analyst's decision
ANALYST_ACTION = "approve"   # "approve" | "reject" | "override"
ANALYST_RATIONALE = (
    "All technical indicators confirm active supply chain compromise. "
    "The reverse shell establishment and credentials exfiltration from 4 CI runners "
    "require immediate IR activation. Approving escalation — rotating secrets in parallel."
)
OVERRIDE_DESCRIPTION = None  # Required if ANALYST_ACTION == "override"

record = dal_engine.submit_decision(
    scenario_id=SCENARIO_ID,
    ai_result=result,
    analyst_action=ANALYST_ACTION,
    analyst_rationale=ANALYST_RATIONALE,
    override_description=OVERRIDE_DESCRIPTION,
)

print(f"Decision submitted.")
print(f"Record ID  : {record.record_id}")
print(f"Timestamp  : {record.timestamp}")
print(f"Action     : {record.analyst_action.upper()}")
print(f"Rationale  : {record.analyst_rationale}")

---
## Step 5 — Audit Record & SHAKE-256 Integrity Hash

This is **Element 5 (Audit Record)** of the Decision Assurance Framework.

The `DecisionRecord` was hashed before saving. The SHAKE-256 digest (post-quantum-resilient,
NIST FIPS 202) was computed over all fields (excluding `record_hash` itself, with keys sorted).
Any post-save modification to any field will produce a different hash — making tampering detectable.

In [ ]:
print(f"SHAKE-256 Hash:\n  {record.record_hash}")
print(f"\nFull record JSON:")
print(record.to_json())

In [ ]:
# Verify the saved record on disk
if cos_client.is_demo_mode():
    saved_path = cos_client.local_decisions_path() / f"{record.record_id}.json"
    saved_dict = json.loads(saved_path.read_text(encoding="utf-8"))
    is_valid = DecisionRecord.verify_record_dict(saved_dict)
    computed = DecisionRecord.compute_hash_for_dict(saved_dict)
    print(f"File         : {saved_path.name}")
    print(f"Integrity    : {'✅ VERIFIED' if is_valid else '❌ HASH MISMATCH'}")
    print(f"Stored hash  : {saved_dict['record_hash']}")
    print(f"Computed hash: {computed}")
    print(f"Match        : {saved_dict['record_hash'] == computed}")
else:
    print("Record saved to IBM COS — verify via GET /audit/{record_id} API endpoint.")

---
## Step 6 — Tamper Simulation

To demonstrate that the tamper-evidence mechanism works, we:
1. Open the saved JSON file
2. Modify the `analyst_rationale` field
3. Recompute the hash from the modified content
4. Observe that it no longer matches the stored hash

This is what the Audit Log's `❌ Hash mismatch` indicator detects.

In [ ]:
if cos_client.is_demo_mode():
    saved_path = cos_client.local_decisions_path() / f"{record.record_id}.json"
    original = json.loads(saved_path.read_text(encoding="utf-8"))
    
    # Tamper with one field
    tampered = dict(original)
    tampered["analyst_rationale"] = "TAMPERED — this was not what the analyst wrote."
    
    stored_hash   = original["record_hash"]
    computed_clean   = DecisionRecord.compute_hash_for_dict(original)
    computed_tampered = DecisionRecord.compute_hash_for_dict(tampered)
    
    print("Stored hash (at save time):")
    print(f"  {stored_hash}")
    print("\nRecomputed hash (clean record):")
    print(f"  {computed_clean}")
    print(f"  Match: {stored_hash == computed_clean}  ✅")
    print("\nRecomputed hash (after tampering analyst_rationale):")
    print(f"  {computed_tampered}")
    print(f"  Match: {stored_hash == computed_tampered}  ❌  <-- tamper detected")
else:
    print("Tamper simulation only available in local demo mode.")

---
## Step 7 — Audit Log

The audit log retrieves all saved records from storage and annotates each with
integrity verification results. Every record shows:
- The stored hash (written at save time)
- The recomputed hash (computed now)
- Whether `integrity_valid` is True or False

In [ ]:
audit_log = dal_engine.get_audit_log()

print(f"{len(audit_log)} record(s) in audit log (newest first):\n")
print(f"{'Record ID':10}  {'Scenario':42}  {'Action':8}  {'Integrity':16}  {'Timestamp':19}")
print("-" * 105)
for r in audit_log:
    integrity = '✅ Verified' if r.get('integrity_valid') else '❌ Hash mismatch'
    ts = r.get('timestamp', '')[:19].replace('T', ' ')
    rid = r.get('record_id', '')[:8] + '...'
    title = r.get('scenario_title', '')[:40]
    action = r.get('analyst_action', '').upper()
    print(f"{rid:10}  {title:42}  {action:8}  {integrity:16}  {ts}")

---
## Step 8 — Working Through All Five DAL Elements

Let's verify we can find all five Decision Assurance Framework elements in
the record we just created.

In [ ]:
rec_dict = record.to_dict()

elements = {
    "1. Evidence Bundle": [
        f"scenario_id    = {rec_dict['scenario_id']}",
        f"scenario_title = {rec_dict['scenario_title']}",
    ],
    "2. AI Recommendation": [
        f"ai_recommendation     = {rec_dict['ai_recommendation']}",
        f"ai_reasoning          = {rec_dict['ai_reasoning'][:80]}...",
        f"ai_suggested_actions  = {rec_dict['ai_suggested_actions']}",
    ],
    "3. Uncertainty Score": [
        f"ai_confidence = {rec_dict['ai_confidence']:.0%}",
    ],
    "4. Human Judgment": [
        f"analyst_id       = {rec_dict['analyst_id']}",
        f"analyst_action   = {rec_dict['analyst_action']}",
        f"analyst_rationale = {rec_dict['analyst_rationale'][:80]}...",
    ],
    "5. Audit Record": [
        f"record_id   = {rec_dict['record_id']}",
        f"timestamp   = {rec_dict['timestamp']}",
        f"record_hash = {rec_dict['record_hash'][:32]}...",
    ],
}

for element, fields in elements.items():
    print(f"\n{'='*60}")
    print(f"  {element}")
    print(f"{'='*60}")
    for f in fields:
        print(f"  {f}")

---
## Summary

This notebook demonstrated the full Decision Assurance Layer workflow:

| Step | DAL Element | What Happened |
|---|---|---|
| Load scenario | Evidence Bundle | Threat fixture loaded from `data/scenarios/` |
| Get recommendation | AI Recommendation | Granite returned action + reasoning |
| Get ranked actions | Uncertainty Score | All three actions scored independently |
| Submit decision | Human Judgment | Analyst approved with documented rationale |
| Save record | Audit Record | SHAKE-256 hashed (PQC-resilient), persisted to disk/COS |
| Verify integrity | Audit Record | Stored hash matched recomputed hash |
| Tamper simulation | Audit Record | Modified field produced different hash |
| Audit log | Audit Record | All records listed with integrity status |

**The key insight:** The DAL does not make decisions. It makes decisions *auditable*.

> *Future cybersecurity is not AI autonomy.
> Future cybersecurity is accountable AI-assisted command and control.*